In [28]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

In [29]:
train_data = pd.read_csv("../dataset/train_metadata.csv")
val_data = pd.read_csv("../dataset/val_metadata.csv")
test_data = pd.read_csv("../dataset/test_metadata.csv")

In [30]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [31]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [32]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [33]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

In [34]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1)

])

In [35]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [36]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [37]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [38]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [39]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
efficientnet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")])
efficientnet_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [40]:
efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [41]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)

In [42]:
history_eff = efficientnet_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.2337 - loss: 1.9746 - val_accuracy: 0.6698 - val_loss: 1.8683
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 288s 1s/step - accuracy: 0.1772 - loss: 1.9638 - val_accuracy: 0.5639 - val_loss: 1.9095
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 289s 1s/step - accuracy: 0.1384 - loss: 1.9625 - val_accuracy: 0.1145 - val_loss: 1.9202
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.1258 - loss: 1.9650 - val_accuracy: 0.1092 - val_loss: 1.9342
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [43]:
train_loss, train_accuracy = efficientnet_model.evaluate(train_dataset)
val_loss, val_accuracy = efficientnet_model.evaluate(val_dataset)
test_loss, test_accuracy = efficientnet_model.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 253s 1s/step - accuracy: 0.6695 - loss: 1.8745
47/47 ━━━━━━━━━━━━━━━━━━━━ 44s 934ms/step - accuracy: 0.6698 - loss: 1.8683
47/47 ━━━━━━━━━━━━━━━━━━━━ 46s 976ms/step - accuracy: 0.6693 - loss: 1.8685


In [25]:
efficientnet_model.save("../models/efficientnet_model.keras")

In [26]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
efficientnet_model_sgd= models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")])
efficientnet_model_sgd.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [44]:
efficientnet_model_sgd.compile(

    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_eff_sgd = efficientnet_model_sgd.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 286s 1s/step - accuracy: 0.1049 - loss: 1.9795 - val_accuracy: 0.0113 - val_loss: 1.9676
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - accuracy: 0.1198 - loss: 1.9697 - val_accuracy: 0.0113 - val_loss: 1.9631
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 326s 1s/step - accuracy: 0.1311 - loss: 1.9732 - val_accuracy: 0.0113 - val_loss: 1.9606
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 427s 2s/step - accuracy: 0.1290 - loss: 1.9693 - val_accuracy: 0.0113 - val_loss: 1.9583
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 423s 2s/step - accuracy: 0.1270 - loss: 1.9731 - val_accuracy: 0.0113 - val_loss: 1.9565
Restoring model weights from the end of the best epoch: 5.


In [45]:
train_loss, train_accuracy = efficientnet_model_sgd.evaluate(train_dataset)
val_loss, val_accuracy = efficientnet_model_sgd.evaluate(val_dataset)
test_loss, test_accuracy = efficientnet_model_sgd.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 349s 2s/step - accuracy: 0.0116 - loss: 1.9533
47/47 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.0113 - loss: 1.9565
47/47 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.0113 - loss: 1.9565


In [46]:
efficientnet_model_sgd.save("../models/efficientnet_model_sgd.keras")

In [47]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
efficientnet_model_RMSprop= models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")])
efficientnet_model_RMSprop.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [48]:
efficientnet_model_RMSprop.compile(

    optimizer=tf.keras.optimizers.RMSprop(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_eff_RMSprop = efficientnet_model_RMSprop.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 417s 2s/step - accuracy: 0.1682 - loss: 1.9737 - val_accuracy: 0.6698 - val_loss: 1.8547
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 416s 2s/step - accuracy: 0.1836 - loss: 1.9613 - val_accuracy: 0.1099 - val_loss: 1.8713
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 312s 1s/step - accuracy: 0.2104 - loss: 1.9626 - val_accuracy: 0.1099 - val_loss: 1.8869
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 275s 1s/step - accuracy: 0.1753 - loss: 1.9554 - val_accuracy: 0.0513 - val_loss: 1.9012
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [49]:
train_loss, train_accuracy = efficientnet_model_RMSprop.evaluate(train_dataset)
val_loss, val_accuracy = efficientnet_model_RMSprop.evaluate(val_dataset)
test_loss, test_accuracy = efficientnet_model_RMSprop.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 240s 1s/step - accuracy: 0.6695 - loss: 1.8562
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 997ms/step - accuracy: 0.6698 - loss: 1.8547
47/47 ━━━━━━━━━━━━━━━━━━━━ 45s 959ms/step - accuracy: 0.6693 - loss: 1.8547


In [53]:
efficientnet_model_RMSprop.save("../models/efficientnet_model_RMSprop.keras")

In [54]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)
 
train_dataset = train_dataset.map(
    apply_augmentation,
    num_parallel_calls=tf.data.AUTOTUNE
)
 
BATCH_SIZE = 64
 
train_dataset = (
    train_dataset
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
 
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [56]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model_64 = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_64.trainable = False

efficient_64= models.Sequential([

    base_model_64,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

efficient_64.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [57]:
efficient_64.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_eff_64 = efficient_64.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 1944s 18s/step - accuracy: 0.0900 - loss: 1.9820 - val_accuracy: 0.0513 - val_loss: 1.9598
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 623s 6s/step - accuracy: 0.1009 - loss: 1.9484 - val_accuracy: 0.0326 - val_loss: 1.9429
Epoch 3/5


I0000 00:00:1785467818.668599  280490 shuffle_dataset_op.cc:453] ShuffleDatasetV3:59: Filling up shuffle buffer (this may take a while): 809 of 1000
I0000 00:00:1785467821.101051  280490 shuffle_dataset_op.cc:483] Shuffle buffer filled.


110/110 ━━━━━━━━━━━━━━━━━━━━ 740s 6s/step - accuracy: 0.1185 - loss: 1.9462 - val_accuracy: 0.0326 - val_loss: 1.9424
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 807s 7s/step - accuracy: 0.0318 - loss: 1.9462 - val_accuracy: 0.0113 - val_loss: 1.9455
Epoch 5/5


I0000 00:00:1785469365.862694  327545 shuffle_dataset_op.cc:453] ShuffleDatasetV3:59: Filling up shuffle buffer (this may take a while): 850 of 1000
I0000 00:00:1785469367.217548  327545 shuffle_dataset_op.cc:483] Shuffle buffer filled.


110/110 ━━━━━━━━━━━━━━━━━━━━ 540s 5s/step - accuracy: 0.0225 - loss: 1.9462 - val_accuracy: 0.0113 - val_loss: 1.9451
Restoring model weights from the end of the best epoch: 3.


In [58]:
train_loss, train_accuracy = efficient_64.evaluate(train_dataset)
val_loss, val_accuracy = efficient_64.evaluate(val_dataset)
test_loss, test_accuracy = efficient_64.evaluate(test_dataset)

I0000 00:00:1785469909.677776  343182 shuffle_dataset_op.cc:453] ShuffleDatasetV3:59: Filling up shuffle buffer (this may take a while): 812 of 1000
I0000 00:00:1785469911.396143  343182 shuffle_dataset_op.cc:483] Shuffle buffer filled.


110/110 ━━━━━━━━━━━━━━━━━━━━ 497s 4s/step - accuracy: 0.0327 - loss: 1.9424
24/24 ━━━━━━━━━━━━━━━━━━━━ 100s 4s/step - accuracy: 0.0326 - loss: 1.9424
24/24 ━━━━━━━━━━━━━━━━━━━━ 98s 4s/step - accuracy: 0.0326 - loss: 1.9424


In [59]:
efficient_64.save("../models/efficient_64.keras")

In [60]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load MobileNetV2

base_model = EfficientNetB0(

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

# Fine-Tuning

base_model.trainable = True

# Freeze all layers except the last 30

for layer in base_model.layers[:-30]:

    layer.trainable = False

# Build Model

eff_ft = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        7,
        activation="softmax"
    )

])

eff_ft.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 1,661,031 (6.34 MB)

 Non-trainable params: 2,553,411 (9.74 MB)

In [61]:
eff_ft.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_eff_ft = eff_ft.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 536s 4s/step - accuracy: 0.1408 - loss: 2.0078 - val_accuracy: 0.0113 - val_loss: 2.1204
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 478s 4s/step - accuracy: 0.1294 - loss: 1.9725 - val_accuracy: 0.0140 - val_loss: 2.0587
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 373s 3s/step - accuracy: 0.1185 - loss: 1.9668 - val_accuracy: 0.0113 - val_loss: 1.9691
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 334s 3s/step - accuracy: 0.1167 - loss: 1.9677 - val_accuracy: 0.0113 - val_loss: 1.9304
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 373s 3s/step - accuracy: 0.1444 - loss: 1.9652 - val_accuracy: 0.0513 - val_loss: 1.9335
Restoring model weights from the end of the best epoch: 4.


In [62]:
train_loss, train_accuracy = eff_ft.evaluate(train_dataset)
val_loss, val_accuracy = eff_ft.evaluate(val_dataset)
test_loss, test_accuracy = eff_ft.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 256s 2s/step - accuracy: 0.0116 - loss: 1.9252
24/24 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.0113 - loss: 1.9304
24/24 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.0113 - loss: 1.9308


In [63]:
efficient_64.save("../models/efficient_64.keras")

In [66]:
import keras_tuner as kt
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, RMSprop

num_classes = 7

def build_model(hp):

    base_model = EfficientNetB0(

        weights="imagenet",

        include_top=False,

        input_shape=(224,224,3)

    )

    base_model.trainable = False

    eff_model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(

            units=hp.Choice(

                "dense_units",

                [128,256,512]

            ),

            activation="relu"

        ),

        layers.Dropout(

            hp.Choice(

                "dropout",

                [0.3,0.5,0.6]

            )

        ),

        layers.Dense(

            num_classes,

            activation="softmax"

        )

    ])

    learning_rate = hp.Choice(

        "learning_rate",

        [1e-3,1e-4,1e-5]

    )

    optimizer = hp.Choice(

        "optimizer",

        ["adam","rmsprop"]

    )

    if optimizer == "adam":

        opt = Adam(

            learning_rate=learning_rate

        )

    else:

        opt = RMSprop(

            learning_rate=learning_rate

        )

    eff_model.compile(

        optimizer=opt,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return eff_model

In [68]:
tuner = kt.RandomSearch(

    build_model,

    objective="val_accuracy",

    max_trials=3,

    directory="eff_tuner",

    project_name="mobilenet_hyperparameter"

)

In [69]:
tuner.search(

    train_dataset,

    validation_data=val_dataset,

    epochs=5,

    class_weight=class_weights

)

Trial 3 Complete [00h 25m 59s]
val_accuracy: 0.6697736382484436

Best val_accuracy So Far: 0.6697736382484436
Total elapsed time: 01h 17m 25s


In [70]:
best_eff_net = tuner.get_best_models(1)[0]

/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [71]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'dense_units': 512, 'dropout': 0.3, 'learning_rate': 0.001, 'optimizer': 'adam'}


In [74]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
efficientnet_final= models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512,activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(7,activation="softmax")])
efficientnet_final.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         3,591 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,709,034 (17.96 MB)

 Trainable params: 659,463 (2.52 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [75]:
efficientnet_final.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True,verbose=1)
history_eff_sgd = efficientnet_final.fit(train_dataset,validation_data=val_dataset,epochs=8,class_weight=class_weights,callbacks=[early_stopping])

Epoch 1/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 307s 3s/step - accuracy: 0.1434 - loss: 2.0129 - val_accuracy: 0.0113 - val_loss: 2.0247
Epoch 2/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 293s 3s/step - accuracy: 0.1039 - loss: 1.9606 - val_accuracy: 0.6698 - val_loss: 1.8928
Epoch 3/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 284s 3s/step - accuracy: 0.0816 - loss: 1.9493 - val_accuracy: 0.0113 - val_loss: 1.9445
Epoch 4/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 308s 3s/step - accuracy: 0.1455 - loss: 1.9462 - val_accuracy: 0.0513 - val_loss: 1.9460
Epoch 5/8
110/110 ━━━━━━━━━━━━━━━━━━━━ 308s 3s/step - accuracy: 0.0451 - loss: 1.9461 - val_accuracy: 0.0513 - val_loss: 1.9457
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [76]:
train_loss, train_accuracy = efficientnet_final.evaluate(train_dataset)
val_loss, val_accuracy = efficientnet_final.evaluate(val_dataset)
test_loss, test_accuracy = efficientnet_final.evaluate(test_dataset)

110/110 ━━━━━━━━━━━━━━━━━━━━ 256s 2s/step - accuracy: 0.6695 - loss: 1.8931
24/24 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.6698 - loss: 1.8928
24/24 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.6693 - loss: 1.8928


In [78]:
efficientnet_final.save("../models/efficientnet_final.keras")